# Bronze Layer — Garmin Data Ingestion
Loads raw Garmin data from SQLite export into a Delta table in the landing volume.

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS garmin_lakehouse;
CREATE SCHEMA IF NOT EXISTS garmin_lakehouse.raw;
CREATE VOLUME IF NOT EXISTS garmin_lakehouse.raw.landing;

In [0]:
%pip install garminconnect

## Bronze Layer — Garmin Activity Ingestion
Connects to Garmin API using credentials stored in Databricks Secrets,
fetches activities and writes them as a Delta table to Unity Catalog.

In [0]:
import pandas as pd
from garminconnect import Garmin
import json

In [0]:
# Pipeline parameters
# Change load_type to "full" for first-time load of a new user
# Change athlete to match the secrets scope for that user
load_type = "incremental"  # "full" or "incremental"
athlete = "bogdan"

In [0]:
# Retrieve credentials from Databricks Secrets
email = dbutils.secrets.get(scope="garmin", key="email")
password = dbutils.secrets.get(scope="garmin", key="password")

print("Credentials loaded ✅")

In [0]:
# Connect to Garmin API
api = Garmin(email=email, password=password)
api.login()

print("Garmin API connected ✅")

In [0]:
from datetime import datetime, timedelta

if load_type == "full":
    all_activities = []
    start = 0
    chunk_size = 100

    while True:
        chunk = api.get_activities(start, chunk_size)
        if not chunk:
            break
        all_activities.extend(chunk)
        start += chunk_size
        print(f"Fetched {len(all_activities)} activities so far...")

    df_new = pd.DataFrame(all_activities)
    print(f"Full load complete: {len(df_new)} activities")

else:
    latest_date = spark.table("garmin_lakehouse.raw.activities") \
        .agg({"startTimeLocal": "max"}) \
        .collect()[0][0]
    
    start_date = (datetime.strptime(latest_date, "%Y-%m-%d %H:%M:%S") - timedelta(days=1)).strftime("%Y-%m-%d")
    end_date = datetime.now().strftime("%Y-%m-%d")
    
    print(f"Incremental load: fetching from {start_date} to {end_date}")
    activities = api.get_activities_by_date(start_date, end_date)
    df_new = pd.DataFrame(activities)
    print(f"Found {len(df_new)} activities")

In [0]:
# Serialise complex fields
for col_name in df_new.columns:
    if df_new[col_name].apply(lambda x: isinstance(x, (dict, list))).any():
        df_new[col_name] = df_new[col_name].apply(
            lambda x: json.dumps(x) if isinstance(x, (dict, list)) else x
        )

# Convert to Spark DataFrame
spark_df = spark.createDataFrame(df_new)

# Align schema with existing table
existing_cols = spark.table("garmin_lakehouse.raw.activities").columns
from pyspark.sql.functions import lit
from pyspark.sql.types import StringType

for col_name in existing_cols:
    if col_name not in spark_df.columns:
        spark_df = spark_df.withColumn(col_name, lit(None).cast(StringType()))

spark_df = spark_df.select(existing_cols)

# Merge into Delta table
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "garmin_lakehouse.raw.activities")

delta_table.alias("existing") \
    .merge(
        spark_df.alias("new"),
        "existing.activityId = new.activityId"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

count = spark.table("garmin_lakehouse.raw.activities").count()
print(f"Total rows: {count} ✅")

In [0]:
# Verify the table
df_check = spark.table("garmin_lakehouse.raw.activities")
print(f"Rows: {df_check.count()}")
print(f"Columns: {len(df_check.columns)}")
df_check.display()